In [ ]:
# ═══════════════════════════════════════════════
# STABLE MONEY AI AVATAR — MASTER CELL
# Delete all other cells, paste this, run once
# ═══════════════════════════════════════════════

import subprocess, os, sys, time

# ── STEP 1: Fix Ollama install ──
print("📦 Installing zstd + Ollama...")
os.system("apt-get install -y zstd > /dev/null 2>&1")
os.system("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1")
print("✅ Ollama installed")

# ── STEP 2: Install Python deps ──
print("📦 Installing Python dependencies...")
os.system("pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles")
os.system("pip install -q 'kokoro>=0.9.2' soundfile")
os.system("apt-get -qq install -y espeak-ng > /dev/null 2>&1")
os.system("pip install -q pyngrok nest_asyncio")
print("✅ Dependencies installed")

# ── STEP 3: Check GPU ──
import torch
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

# ── STEP 4: Configure ngrok ──
print("🔗 Configuring ngrok...")
NGROK_TOKEN = "3BO5O8HXbvLr8j8TjHYPj5UFm4r_3BYg4BqamLqx7t7et34y9"
from pyngrok import conf, ngrok
conf.get_default().auth_token = NGROK_TOKEN
os.system(f"ngrok config add-authtoken {NGROK_TOKEN}")
print("✅ ngrok configured")

# ── STEP 5: Start Ollama + pull model ──
print("🤖 Starting Ollama...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("📥 Pulling llama3.2:3b (2-3 mins)...")
os.system("ollama pull llama3.2:3b")
print("✅ Ollama ready")

# ── STEP 6: Write server.py ──
print("📝 Writing server.py...")
server_code = '''
import asyncio, base64, io, os
from pathlib import Path
import torch
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from kokoro import KPipeline
import soundfile as sf
import numpy as np

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[server] Device: {DEVICE}")

class Models:
    tts = None
models = Models()

@app.on_event("startup")
async def load_models():
    models.tts = KPipeline(lang_code="a")
    print("[startup] Kokoro TTS loaded")

def synth(text):
    generator = models.tts(text, voice="af_heart", speed=1.1)
    chunks = [audio for _, _, audio in generator]
    audio_data = np.concatenate(chunks)
    buf = io.BytesIO()
    sf.write(buf, audio_data, 24000, format="WAV")
    buf.seek(0)
    return buf.read()

@app.websocket("/ws/talk")
async def ws_talk(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_json()
            if data.get("type") in ("config", None): continue
            if data.get("type") != "speak": continue
            text = data.get("text", "").strip()
            if not text: continue
            prompt = f"""You are Aayush, a warm advisor for Stable Money India.
Stable Money offers FDs from 25+ NBFCs with 8-9.5% returns. Sukoon: no-lock-in 7-8%. Min Rs 1000. DICGC insured.
Answer in 2 sentences max. Warm and simple. No links.
Question: {text}"""
            try:
                import urllib.request as ur, json
                req = ur.Request("http://localhost:11434/api/generate",
                    data=json.dumps({"model":"llama3.2:3b","prompt":prompt,"stream":False,
                        "options":{"num_predict":60,"temperature":0.7}}).encode(),
                    headers={"Content-Type":"application/json"})
                with ur.urlopen(req, timeout=30) as r:
                    answer = json.loads(r.read()).get("response","").strip()
            except Exception as e:
                answer = f"Sorry, I had a hiccup. Please try again!"
            await ws.send_json({"type":"text_chunk","text":answer})
            try:
                audio = await asyncio.get_event_loop().run_in_executor(None, synth, answer)
                await ws.send_json({"type":"audio","data":base64.b64encode(audio).decode(),"format":"wav"})
            except Exception as e:
                await ws.send_json({"type":"error","msg":str(e)})
            await ws.send_json({"type":"done"})
    except WebSocketDisconnect:
        pass

@app.get("/health")
def health():
    return {"status":"ok","device":DEVICE}

if Path("./static").exists():
    app.mount("/", StaticFiles(directory="static", html=True), name="static")
'''
with open('/content/server.py', 'w') as f:
    f.write(server_code)
print("✅ server.py written")

# ── STEP 7: Download frontend ──
print("🌐 Downloading frontend...")
os.makedirs('/content/static', exist_ok=True)
os.system("curl -s https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main/static/index.html -o /content/static/index.html")
size = os.path.getsize('/content/static/index.html')
print(f"✅ Frontend downloaded ({size} bytes)")

# ── STEP 8: Download avatar photo from GitHub ──
print("📸 Downloading avatar from GitHub...")
os.system("curl -s https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main/static/avatar.jpg -o /content/avatar.jpg")
size = os.path.getsize('/content/avatar.jpg')
print(f"✅ Avatar downloaded ({size} bytes)")

# ── STEP 9: Start server + ngrok ──
print("🚀 Starting server...")
import nest_asyncio, uvicorn
nest_asyncio.apply()
os.chdir('/content')
sys.path.insert(0, '/content')

ngrok.kill()
time.sleep(2)
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
ws_url = public_url.replace('https://', 'wss://') + '/ws/talk'

print('=' * 50)
print('🌐  PUBLIC URL:', public_url)
print('🔌  WS URL:', ws_url)
print('=' * 50)

config = uvicorn.Config('server:app', host='0.0.0.0', port=8000, log_level='info')
server_inst = uvicorn.Server(config)
await server_inst.serve()